# NB05: Full Evaluation

First and only use of DS2 / fold 10.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab — skip Drive mount.')

## RUN MODE — edit the cell below (only switch in this notebook)

| `XFAD_DRY_RUN` | Meaning |
|---|---|
| `'1'` | **Smoke test** — tiny samples, 1 epoch, fast sanity check (~minutes) |
| `'0'` | **Full paper run** — all data, full training, both student arms (~hours) |

On a **full run**, NB02–NB05 automatically process **baseline** and **saliency** students (everything NB06 needs).

Then: **Runtime → Run all** (restart session after changing the switch).


In [ ]:
import os

# ╔══════════════════════════════════════════════════════════════════════╗
# ║  ONLY SWITCH IN THIS NOTEBOOK                                        ║
# ║  '0' = FULL PAPER RUN (all data, both student arms)                  ║
# ║  '1' = SMOKE TEST (tiny data, baseline student only)                 ║
# ╚══════════════════════════════════════════════════════════════════════╝
os.environ['XFAD_DRY_RUN'] = '0'


In [ ]:
import os
import sys
import json
from pathlib import Path


def bootstrap_xfad_root() -> Path:
    """Locate repo root and put it on sys.path (works in Colab + local)."""
    candidates = []
    env = os.environ.get('XFAD_ROOT')
    if env:
        candidates.append(Path(env))
    candidates.extend([
        Path('/content/drive/MyDrive/XFAD'),
        Path.cwd(),
        Path.cwd().parent,
    ])
    candidates.extend(Path.cwd().resolve().parents)
    seen = set()
    for p in candidates:
        try:
            p = p.resolve()
        except OSError:
            continue
        if p in seen:
            continue
        seen.add(p)
        if (p / 'src' / 'config.py').exists():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            os.environ.setdefault('XFAD_ROOT', str(p))
            return p
    raise FileNotFoundError(
        'XFAD repo root not found (expected src/config.py). '
        'In Colab: mount Drive first, then either keep the project at '
        '/content/drive/MyDrive/XFAD or set '
        "os.environ['XFAD_ROOT'] = '/content/drive/MyDrive/<your>/XFAD'."
    )


REPO_ROOT = bootstrap_xfad_root()
print('REPO_ROOT:', REPO_ROOT)

from src.utils.io import set_seed
set_seed(42)

from src.utils.io import resolve_dry_run, print_run_banner, ensure_dir, student_variants_for_run
DRY_RUN = resolve_dry_run()

try:
    import wfdb
except ImportError:
    import subprocess
    print('Installing wfdb...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb', 'tqdm', 'joblib'])
    import wfdb
print('wfdb version:', wfdb.__version__)

import numpy as np
import torch
import torch.nn.functional as F
from src.config import MITBIH_DS2, PTBXL_TEST_FOLD, MODEL_DIR, RESULTS_DIR, SENSITIVITY_LOSS_BUDGET, STUDENT_TRUNK_CHANNELS
from src.data.mitbih import MITBIHDataset
from src.data.ptbxl import PTBXLDataset
from src.metrics.bootstrap import patient_bootstrap_ci
from src.metrics.significance import mcnemar_test, holm_bonferroni
from src.models.student import MultiTaskStudent
from src.models.teacher_a import TeacherA
from src.models.teacher_b import TeacherB

device = torch.device('cuda' if torch.cuda.is_available() and not DRY_RUN else 'cpu')
ensure_dir(RESULTS_DIR)

STUDENT_VARIANTS_TO_RUN = student_variants_for_run(DRY_RUN)
print_run_banner(notebook='NB05_Full_Evaluation', dry_run=DRY_RUN, variants=STUDENT_VARIANTS_TO_RUN)

In [ ]:
# Load test sets (DS2 / fold 10) - first time in pipeline
if not DRY_RUN:
    from src.data.cache import build_mitbih_cache, build_ptbxl_cache
    # require_groups rebuilds any cache saved before record/patient ids were tracked,
    # which patient-level bootstrap CIs need.
    for label, build in (
        ('MIT DS2 cache', lambda strict: build_mitbih_cache(MITBIH_DS2, dry_run=False, require_groups=strict)),
        ('PTB fold-10 cache', lambda strict: build_ptbxl_cache([PTBXL_TEST_FOLD], dry_run=False, require_groups=strict)),
    ):
        try:
            print(label + ':', build(True))
        except Exception as exc:  # raw data unavailable in this session
            print(f'{label}: could not rebuild with group ids ({exc})')
            print(label + ':', build(False))

mit_test = MITBIHDataset(MITBIH_DS2, dry_run=DRY_RUN, use_cache=True)
ptb_test = PTBXLDataset([PTBXL_TEST_FOLD], dry_run=DRY_RUN, use_cache=True)
print('Test sizes:', len(mit_test), len(ptb_test))
print('Record ids in MIT-BIH cache:', bool(mit_test.groups),
      '| patient ids in PTB-XL cache:', bool(ptb_test.groups))
if not (mit_test.groups and ptb_test.groups):
    print('WARNING: missing group ids - confidence intervals fall back to sample level')


In [ ]:
for VARIANT in STUDENT_VARIANTS_TO_RUN:
    print('\n' + '=' * 60 + f'\nStudent variant: {VARIANT}\n' + '=' * 60)
    from src.utils.io import student_checkpoint_path, variant_result_path
    STUDENT_PATH = student_checkpoint_path(VARIANT)
    if not os.path.exists(STUDENT_PATH):
        raise FileNotFoundError(f'{STUDENT_PATH} missing — run NB02 first (full run trains both arms).')
    print(f'Checkpoint: {STUDENT_PATH}')

    student = MultiTaskStudent(c_internal=STUDENT_TRUNK_CHANNELS).to(device)
    student.load_state_dict(torch.load(STUDENT_PATH, map_location=device))
    teacher_a = TeacherA().to(device)
    teacher_b = TeacherB().to(device)
    teacher_a.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'teacher_a.pt'), map_location=device))
    teacher_b.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'teacher_b.pt'), map_location=device))
    student.eval(); teacher_a.eval(); teacher_b.eval()

    from sklearn.metrics import f1_score, recall_score
    from tqdm.auto import tqdm
    from src.config import AAMI_CLASS_TO_IDX, MITBIH_NORMAL_CLASS, COMMON_FS
    from src.metrics.significance import delong_roc_test
    from src.gate.evaluation import collect_alarm_records, evaluate_baselines, format_results_table

    def predict_dataset(dataset, task, teacher, *, n_max=None, batch_size=128):
        n = len(dataset) if n_max is None else min(len(dataset), n_max)
        groups_available = bool(getattr(dataset, 'groups', []))
        y, probs_s, probs_t, groups = [], [], [], []
        with torch.no_grad():
            for start in tqdm(range(0, n, batch_size), desc=f'predict task {task}'):
                idx = list(range(start, min(start + batch_size, n)))
                xb = torch.stack([dataset[i][0] for i in idx]).to(device)
                y += [int(dataset[i][1]) for i in idx]
                probs_s.append(F.softmax(student(xb, task), dim=1).cpu().numpy())
                probs_t.append(F.softmax(teacher(xb), dim=1).cpu().numpy())
                groups += [str(dataset.groups[i]) if groups_available else f'sample_{i}' for i in idx]
        return np.array(y), np.vstack(probs_s), np.vstack(probs_t), np.array(groups), groups_available

    def classification_metrics(name, y, probs_s, probs_t, groups, normal_index, *, groups_available, n_boot):
        pred_s, pred_t = probs_s.argmax(1), probs_t.argmax(1)
        y_bin = (y != normal_index).astype(int)
        score_s = 1.0 - probs_s[:, normal_index]
        score_t = 1.0 - probs_t[:, normal_index]
        out = {
            'dataset': name,
            'n_samples': int(len(y)),
            'n_groups': int(len(np.unique(groups))),
            'patient_level_ci': bool(groups_available),
            'student': {
                'accuracy': float((pred_s == y).mean()),
                'macro_f1': float(f1_score(y, pred_s, average='macro', zero_division=0)),
                'abnormal_recall': float(recall_score(y_bin, (pred_s != normal_index).astype(int), zero_division=0)),
            },
            'teacher': {
                'accuracy': float((pred_t == y).mean()),
                'macro_f1': float(f1_score(y, pred_t, average='macro', zero_division=0)),
                'abnormal_recall': float(recall_score(y_bin, (pred_t != normal_index).astype(int), zero_division=0)),
            },
        }
        if len(np.unique(y_bin)) > 1:
            delong = delong_roc_test(y_bin, score_s, score_t)
            p_mcnemar = mcnemar_test(y, pred_s, pred_t)
            out['student']['auroc_abnormal'] = delong['auc_a']
            out['teacher']['auroc_abnormal'] = delong['auc_b']
            out['delong'] = delong
            out['mcnemar_accuracy_p'] = p_mcnemar
            tests = [('mcnemar_accuracy', p_mcnemar)]
            if delong['p_value'] is not None:
                tests.append(('delong_auroc', delong['p_value']))
            out['holm_bonferroni'] = holm_bonferroni(tests)
        accuracy_fn = lambda a, b: float((a == b).mean())
        for model_key, pred in (('student', pred_s), ('teacher', pred_t)):
            mean, lo, hi = patient_bootstrap_ci(y, pred, groups, accuracy_fn, n_boot=n_boot)
            out[model_key]['accuracy_mean_ci'] = [mean, lo, hi]
        return out

    N_BOOT = 100 if DRY_RUN else 1000
    mit_normal_idx = AAMI_CLASS_TO_IDX[MITBIH_NORMAL_CLASS]
    y_m, ps_m, pt_m, g_m, has_g_m = predict_dataset(mit_test, 'A', teacher_a, n_max=200 if DRY_RUN else None)
    mit_metrics = classification_metrics('mitbih_ds2', y_m, ps_m, pt_m, g_m, mit_normal_idx, groups_available=has_g_m, n_boot=N_BOOT)
    print(json.dumps(mit_metrics, indent=2))

    ptb_normal_idx = PTBXLDataset.normal_class_idx()
    y_p, ps_p, pt_p, g_p, has_g_p = predict_dataset(ptb_test, 'B', teacher_b, n_max=100 if DRY_RUN else None)
    ptb_metrics = classification_metrics('ptbxl_fold10', y_p, ps_p, pt_p, g_p, ptb_normal_idx, groups_available=has_g_p, n_boot=N_BOOT)
    print(json.dumps(ptb_metrics, indent=2))

    gate_path = variant_result_path('gate_thresholds', VARIANT)
    if os.path.exists(gate_path):
        with open(gate_path) as f:
            gate_cfg = json.load(f)
        print('Gate config loaded:', json.dumps(gate_cfg, indent=2))
    else:
        gate_cfg = {'low_threshold': 0.3, 'high_threshold': 0.7, 'note': 'run NB04 first'}
        print('Gate config missing:', gate_cfg)

    normal_idx = AAMI_CLASS_TO_IDX[MITBIH_NORMAL_CLASS]
    gate_model = None
    gate_model_path = variant_result_path('gate_model', VARIANT, ext='.joblib')
    if os.path.exists(gate_model_path):
        import joblib
        gate_model = joblib.load(gate_model_path)
        print('Stage-A scorer: logistic regression from NB04')
    else:
        print(f'Stage-A scorer: fallback risk rule (no {os.path.basename(gate_model_path)} from NB04)')

    low = gate_cfg['low_threshold']
    high = gate_cfg['high_threshold']
    div_thr = gate_cfg.get('divergence_threshold', 0.5)
    baseline_cfg = gate_cfg.get('baseline_calibration', {})
    sqi_thr = baseline_cfg.get('sqi_only', {}).get('threshold', 0.5)
    conf_thr = baseline_cfg.get('confidence_only', {}).get('threshold', 0.8)

    records, cams = collect_alarm_records(
        mit_test, student, normal_idx=normal_idx, fs=COMMON_FS, device=device,
        student_layer=MultiTaskStudent.LAST_CONV, teacher=teacher_a, teacher_layer=TeacherA.LAST_CONV,
        with_cams=True, max_alarms=int(os.environ.get('XFAD_GATE_EVAL_MAX_ALARMS', '0')), progress=True,
    )
    gate_results = evaluate_baselines(
        records, gate_model=gate_model, low=low, high=high, divergence_threshold=div_thr,
        normal_idx=normal_idx, cams=cams, budget=SENSITIVITY_LOSS_BUDGET, n_boot=N_BOOT,
        sqi_threshold=sqi_thr, confidence_threshold=conf_thr,
    )
    print(format_results_table(gate_results, budget=SENSITIVITY_LOSS_BUDGET))

    gate_eval = {
        'dry_run': DRY_RUN,
        'student_variant': VARIANT,
        'dataset': 'mitbih_ds2',
        'population': 'student alarm events (prediction != normal class)',
        'thresholds': {'low': low, 'high': high, 'divergence': div_thr},
        'baseline_calibration': baseline_cfg,
        'lr_model_fitted': gate_model is not None,
        'sensitivity_budget': SENSITIVITY_LOSS_BUDGET,
        'n_groups': len({r.group for r in records}),
        'policies': {name: m.to_dict() for name, m in gate_results.items()},
    }
    with open(variant_result_path('gate_eval_ds2', VARIANT), 'w', encoding='utf-8') as f:
        json.dump(gate_eval, f, indent=2)

    classification_eval = {
        'dry_run': DRY_RUN,
        'student_variant': VARIANT,
        'sensitivity_budget': SENSITIVITY_LOSS_BUDGET,
        'mitbih_ds2': mit_metrics,
        'ptbxl_fold10': ptb_metrics,
    }
    with open(variant_result_path('classification_eval', VARIANT), 'w', encoding='utf-8') as f:
        json.dump(classification_eval, f, indent=2)

# Paper tables: run once after all variants finish (Table 4 needs both arms).
import src.paper.tables as paper_tables
from src.paper.tables import build_all_tables

_orig_ablation_table = paper_tables.ablation_table

def _safe_ablation_table(baseline, saliency):
    baseline = {**baseline, 'classification': baseline.get('classification') or {}}
    saliency = {**saliency, 'classification': saliency.get('classification') or {}}
    return _orig_ablation_table(baseline, saliency)

paper_tables.ablation_table = _safe_ablation_table

for name, text in build_all_tables(RESULTS_DIR, variant='baseline').items():
    print(f'\n===== {name} =====\n{text}')


### Export paper artifacts

Writes `classification_eval.json` (both datasets) next to `gate_eval_ds2.json`, then
renders the paper tables with `scripts/export_paper_tables.py`.
